In Class Assignment 2026.04.14

In [7]:
pip install optuna

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 20.9 MB/s  0:00:00

   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Importing libraries, preparing data, initial EDA

In [8]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


In [9]:
# importing data
adult = pd.read_csv('adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [10]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions

From the EDA, the dataset does not contain missing values, so no imputation is required. Several features show meaningful relationships with the target, 
while others may be less informative.

Based on these characteristics, XGBoost is an appropriate model for this tabular dataset. As a boosting method, 
it builds models sequentially by correcting previous errors, making it effective for capturing complex patterns in the data.

Finally, hyperparameter tuning (e.g., learning rate, max depth) will be applied to control overfitting and improve performance.



### Data Preprocessing (minimal) and Baseline Model

In [11]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [12]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [13]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

The baseline XGBoost model achieved a mean F1 score of approximately 0.71 using stratified cross-validation. This indicates a reasonable performance for the initial model, showing that the model can capture important patterns in the data.

To improve performance, hyperparameter tuning will be applied, particularly adjusting parameters such as learning rate, max depth, and subsample. In addition, techniques such as class imbalance handling (e.g., scale_pos_weight) may further improve F1 score. Feature selection and engineering may also help reduce noise and enhance model performance.

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [15]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# calculate scale_pos_weight from class distribution
neg = (y == 0).sum()
pos = (y == 1).sum()
scale_pos_weight = neg / pos

# models to compare
models = {
    "baseline": XGBClassifier(
        enable_categorical=True,
        random_state=42
    ),
    
    "max_depth_3": XGBClassifier(
        enable_categorical=True,
        random_state=42,
        max_depth=3
    ),
    
    "learning_rate_0.05": XGBClassifier(
        enable_categorical=True,
        random_state=42,
        learning_rate=0.05
    ),
    
    "scale_pos_weight": XGBClassifier(
        enable_categorical=True,
        random_state=42,
        scale_pos_weight=scale_pos_weight
    ),
    
    "combined_model": XGBClassifier(
        enable_categorical=True,
        random_state=42,
        max_depth=3,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        subsample=0.8
    )
}

# evaluate all models
results = []

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring='f1')
    results.append({
        "model": name,
        "mean_f1": scores.mean(),
        "std_f1": scores.std()
    })
    print(f"{name}:")
    print(f"F1 scores = {scores}")
    print(f"Mean F1 = {scores.mean():.4f}")
    print("-" * 40)

# summary table
results_df = pd.DataFrame(results).sort_values(by="mean_f1", ascending=False)
print("\nModel comparison:")
print(results_df)

# identify best model
best_model_name = results_df.iloc[0]["model"]
best_score = results_df.iloc[0]["mean_f1"]

print(f"\nBest model: {best_model_name}")
print(f"Best mean F1 score: {best_score:.4f}")

baseline:
F1 scores = [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 = 0.7120
----------------------------------------
max_depth_3:
F1 scores = [0.70483758 0.72353354 0.70412684 0.71663534 0.71210341]
Mean F1 = 0.7122
----------------------------------------
learning_rate_0.05:
F1 scores = [0.69214235 0.69920271 0.6962963  0.70189045 0.68999028]
Mean F1 = 0.6959
----------------------------------------
scale_pos_weight:
F1 scores = [0.71368994 0.71248651 0.71490751 0.7230658  0.70864729]
Mean F1 = 0.7146
----------------------------------------
combined_model:
F1 scores = [0.68669382 0.68304252 0.68217054 0.69239966 0.68328593]
Mean F1 = 0.6855
----------------------------------------

Model comparison:
                model   mean_f1    std_f1
3    scale_pos_weight  0.714559  0.004743
1         max_depth_3  0.712247  0.007314
0            baseline  0.711967  0.006274
2  learning_rate_0.05  0.695904  0.004382
4      combined_model  0.685518  0.003770

Best model: scal

Different XGBoost hyperparameters were explored using stratified cross-validation to evaluate their impact on model performance.

The baseline model achieved a mean F1 score of approximately 0.712. Adjusting max_depth resulted in a similar performance, 
indicating that tree depth has limited impact in this case. Reducing the learning_rate decreased performance, likely due to slower learning and underfitting.

Using scale_pos_weight improved the F1 score to approximately 0.715, making it the best-performing model. 
This suggests that handling class imbalance is important for this dataset. The combined model did not perform well, possibly due to suboptimal parameter combinations.

Overall, the model with scale_pos_weight performed best, highlighting the importance of addressing class imbalance in boosting models.

### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [16]:
# tuning XGBoost with GridSearchCV

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import f1_score, classification_report

# stratified cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# class imbalance ratio
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

# base model
xgb_model = XGBClassifier(
    enable_categorical=True,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss'
)

# parameter grid
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# grid search
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='f1',
    cv=skf,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

# best model
best_xgb = grid_search.best_estimator_

# predictions on test set
y_pred = best_xgb.predict(X_test)

# results
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation F1:", grid_search.best_score_)
print("Test F1 score:", f1_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 1.0}
Best cross-validation F1: 0.7168996585871618
Test F1 score: 0.7198120028922632

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.84      0.89      7431
           1       0.62      0.85      0.72      2338

    accuracy                           0.84      9769
   macro avg       0.79      0.84      0.80      9769
weighted avg       0.87      0.84      0.85      9769



GridSearchCV was used to tune the XGBoost model using stratified cross-validation and F1 score. 
The best parameters included max_depth=7, learning_rate=0.1, and n_estimators=200.

The tuned model achieved a cross-validation F1 score of about 0.717 and a test F1 score of 0.719, 
slightly improving over the baseline. This shows that hyperparameter tuning helps improve boosting model performance.

### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [17]:
# tuning xgboost classifier with RandomizedSearchCV (tune more parameters than shown here)
param_dist = {
    'scale_pos_weight': np.linspace(1.0, 10.0, num=10),
    'max_depth': np.arange(3, 11),
    'learning_rate': np.linspace(0.01, 0.3, num=10)
}

# replace this placeholder model with your preferred model from above

xgb_random = RandomizedSearchCV(
    XGBClassifier(
        random_state=42,
        enable_categorical=True,
        scale_pos_weight=scale_pos_weight  
    ),
    param_distributions=param_dist,
    n_iter=20,
    cv=skf,
    scoring='f1',
    random_state=42
)

xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')   

# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set

xgb_random_best = XGBClassifier(random_state=42, scale_pos_weight=xgb_random.best_params_['scale_pos_weight'], 
                                max_depth=xgb_random.best_params_['max_depth'], 
                                learning_rate=xgb_random.best_params_['learning_rate'], 
                                enable_categorical=True)
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


Best parameters from RandomizedSearchCV: {'scale_pos_weight': np.float64(2.0), 'max_depth': np.int64(9), 'learning_rate': np.float64(0.23555555555555557)}
Best F1 score from RandomizedSearchCV: 0.7157810076446193
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.92      0.89      0.91      7431
           1       0.69      0.76      0.72      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.83      0.81      9769
weighted avg       0.87      0.86      0.86      9769



RandomizedSearchCV was used to tune the XGBoost model using stratified cross-validation and F1 score. The best parameters included scale_pos_weight=2, max_depth=9, and learning_rate≈0.236.

The tuned model achieved a cross-validation F1 score of about 0.716. The classification report shows strong performance on the majority class and reasonable performance on the minority class, indicating effective handling of class imbalance.

### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [19]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    
    # replace this placeholder model with your preferred model from above
    
    xgb_optuna = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, 
                               max_depth=max_depth,  learning_rate=learning_rate, enable_categorical=True)
    
    cv_scores = cross_val_score(xgb_optuna, X_train, y_train, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42, scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'], 
                                  enable_categorical=True)

xgb_optuna_best = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    scale_pos_weight=study.best_params['scale_pos_weight'],
    max_depth=study.best_params['max_depth'],
    learning_rate=study.best_params['learning_rate']
)

xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


[I 2026-04-15 19:40:34,268] A new study created in memory with name: no-name-01b49f88-d718-462e-b8a6-db574102ac11


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-04-15 19:40:35,567] Trial 0 finished with value: 0.7195493004044421 and parameters: {'scale_pos_weight': 2.781661846672982, 'max_depth': 6, 'learning_rate': 0.2903897427303834}. Best is trial 0 with value: 0.7195493004044421.
[I 2026-04-15 19:40:36,969] Trial 1 finished with value: 0.6833629953113768 and parameters: {'scale_pos_weight': 9.170617220502676, 'max_depth': 7, 'learning_rate': 0.2968465929620905}. Best is trial 0 with value: 0.7195493004044421.
[I 2026-04-15 19:40:38,167] Trial 2 finished with value: 0.6830592339501363 and parameters: {'scale_pos_weight': 7.421052670400746, 'max_depth': 6, 'learning_rate': 0.29228018142877094}. Best is trial 0 with value: 0.7195493004044421.
[I 2026-04-15 19:40:39,295] Trial 3 finished with value: 0.5998267390126546 and parameters: {'scale_pos_weight': 7.373037389943011, 'max_depth': 5, 'learning_rate': 0.011005842874667809}. Best is trial 0 with value: 0.7195493004044421.
[I 2026-04-15 19:40:40,390] Trial 4 finished with value: 0.66

Optuna was used to tune the XGBoost model using stratified cross-validation and F1 score. The best parameters included scale_pos_weight≈2.09 and max_depth=7.

The tuned model achieved a cross-validation F1 score of about 0.720, which was slightly better than the previous tuning methods. This shows that Optuna was effective in finding a strong set of hyperparameters efficiently.

### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?

Three tuning methods were explored: GridSearchCV, RandomizedSearchCV, and Optuna. Among them, Optuna produced the best performance with an F1 score of approximately 0.720, slightly outperforming GridSearchCV (~0.719) and RandomizedSearchCV (~0.716).

GridSearchCV was more exhaustive but computationally expensive, while RandomizedSearchCV was faster but less precise. Optuna provided a good balance by efficiently searching the parameter space and finding strong hyperparameters.

Overall, I prefer Optuna because it achieved the best performance with fewer evaluations, making it both effective and efficient.
